In [3]:
# solver.py
import numpy as np
import pandas as pd
import math
from scipy.sparse import lil_matrix, linalg

# Modificación de la función base para ser más robusta, usar índices 1-based como en el código base,
# y enfocarse solo en el cálculo y retorno de los resultados.
def solve_pde_fourier(m, n, a, b, c, d, f, g, sol_exacta):
    """
    Resuelve la EDP Elíptica bidimensional (Laplace o Poisson) usando el Método de Diferencias Finitas (MDF).

    Args:
        m: (int) Número de intervalos en y (filas internas).
        n: (int) Número de intervalos en x (columnas internas).
        a, b, c, d: (float) Extremos del dominio [a, b] x [c, d].
        f: (fun) Función del lado derecho f(x, y) de la EDP.
        g: (fun) Función para las Condiciones de Frontera g(x, y).
        sol_exacta: (fun) Función para la Solución Analítica exacta V(x, y).

    Returns:
        pd.DataFrame: Con columnas 'x', 'y', 'V_num', 'V_ana', 'error_abs', 'error_rel'.
    """

    # Número de puntos interiores: (n-1) en x, (m-1) en y.
    num_puntos_x = n - 1
    num_puntos_y = m - 1
    total_incognitas = num_puntos_x * num_puntos_y

    h = (b - a) / n  # Paso en x
    k = (d - c) / m  # Paso en y

    # Generación de coordenadas de los puntos interiores
    x = np.array([a + i * h for i in range(1, n)]) # n-1 puntos
    y = np.array([c + j * k for j in range(1, m)]) # m-1 puntos

    # Constantes
    Lambda = (h / k)**2
    mu = 2 * (1 + Lambda)

    # Inicialización de matriz dispersa y vector B
    A = lil_matrix((total_incognitas, total_incognitas))
    B = np.zeros(total_incognitas)

    # Llenado de la matriz A y el vector B
    for l in range(total_incognitas):
        # Indice i (en x, 1 a n-1) y j (en y, 1 a m-1) del punto (x_i, y_j)
        # La indexación va de 1 a n-1 para x, y de 1 a m-1 para y.
        # Usamos el mapeo k -> (i, j) donde k es el índice en el sistema lineal (0-based)
        # y i, j son los índices de la cuadrícula (1-based).
        i = (l % num_puntos_x) + 1  # 1-based index for x
        j = (l // num_puntos_x) + 1  # 1-based index for y

        # Coordenadas reales
        xi = x[i - 1]
        yj = y[j - 1]

        # Lado derecho B
        B[l] = -h**2 * f(xi, yj)

        # Diagonal principal
        A[l, l] = mu

        # Vecinos en x (i-1 y i+1)
        if i > 1:
            A[l, l - 1] = -1
        if i < num_puntos_x:
            A[l, l + 1] = -1

        # Vecinos en y (j-1 y j+1)
        if j > 1:
            A[l, l - num_puntos_x] = -Lambda
        if j < num_puntos_y:
            A[l, l + num_puntos_x] = -Lambda

        # Condiciones de frontera (suma al vector B)
        # Borde izquierdo (i=1, x=a)
        if i == 1:
            B[l] += g(a, yj)
        # Borde derecho (i=n-1, x=b)
        if i == num_puntos_x:
            B[l] += g(b, yj)
        # Borde inferior (j=1, y=c)
        if j == 1:
            B[l] += Lambda * g(xi, c)
        # Borde superior (j=m-1, y=d)
        if j == num_puntos_y:
            B[l] += Lambda * g(xi, d)

    # Solución del sistema lineal
    V_num = linalg.spsolve(A.tocsr(), B)

    # Cálculo de la solución analítica y el error
    V_ana = np.array([sol_exacta(x[i - 1], y[j - 1])
                      for j in range(1, m)
                      for i in range(1, n)])

    error_abs = np.abs(V_ana - V_num)
    error_rel = error_abs / np.abs(V_ana)
    error_rel[np.isinf(error_rel)] = 0 # Manejo de posible división por cero o valor muy pequeño
    error_rel[np.isnan(error_rel)] = 0

    # Crear DataFrame de resultados
    X_coords = np.array([x[i - 1] for j in range(1, m) for i in range(1, n)])
    Y_coords = np.array([y[j - 1] for j in range(1, m) for i in range(1, n)])

    resultados = pd.DataFrame({
        "x": X_coords,
        "y": Y_coords,
        "V_num": V_num,
        "V_ana": V_ana,
        "error_abs": error_abs,
        "error_rel": error_rel
    })

    return resultados


# --- Definiciones de Problemas ---

# Problema 1: V(x, y) = exp(xy)
def run_problema_1(N):
    a, b = 0.0, 2.0  # Dominio x
    c, d = 0.0, 2.0  # Dominio y
    m, n = N, N

    # Solución Analítica
    def V_ana(x, y):
        return np.exp(x * y)

    # Lado Derecho f(x, y) = (x^2 + y^2) * exp(xy)
    def f(x, y):
        return (x**2 + y**2) * np.exp(x * y)

    # Condiciones de Frontera g(x, y)
    def g(x, y):
        if x == a: return 1.0  # V(0, y) = e^(0) = 1
        if x == b: return np.exp(2 * y) # V(2, y) = e^(2y)
        if y == c: return 1.0  # V(x, 0) = e^(0) = 1
        if y == d: return np.exp(2 * x) # V(x, 2) = e^(2x)
        return 0 # Caso general, no debería ocurrir en los bordes

    return solve_pde_fourier(m, n, a, b, c, d, f, g, V_ana)


# Problema 2: V(x, y) = ln(y^2 + x^2)
def run_problema_2(N):
    a, b = 1.0, 2.0
    c, d = 0.0, 2.0
    m, n = N, N

    def V_ana(x, y):
        return np.log(y**2 + x**2)

    # Lado Derecho f(x, y) = 0 (Laplace)
    def f(x, y):
        return 0.0

    # Condiciones de Frontera g(x, y)
    def g(x, y):
        if x == a: return np.log(y**2 + 1**2) # V(1, y) = ln(y^2+1)
        if x == b: return np.log(y**2 + 2**2) # V(2, y) = ln(y^2+4)
        if y == c: return np.log(0**2 + x**2) # V(x, 0) = 2*ln(x) = ln(x^2)
        if y == d: return np.log(2**2 + x**2) # V(x, 2) = ln(4+x^2)
        return 0

    return solve_pde_fourier(m, n, a, b, c, d, f, g, V_ana)


# Problema 3: V(x, y) = (x - y)^2
def run_problema_3(N):
    a, b = 1.0, 2.0
    c, d = 0.0, 2.0
    m, n = N, N

    def V_ana(x, y):
        return (x - y)**2

    # Lado Derecho f(x, y) = 4 (Poisson)
    def f(x, y):
        return 4.0

    # Condiciones de Frontera g(x, y)
    def g(x, y):
        if x == a: return (1 - y)**2 # V(1, y)
        if x == b: return (2 - y)**2 # V(2, y)
        if y == c: return (x - 0)**2 # V(x, 0) = x^2
        if y == d: return (x - 2)**2 # V(x, 2) = (x-2)^2
        return 0

    return solve_pde_fourier(m, n, a, b, c, d, f, g, V_ana)


# Problema 4: V(x, y) = xy * ln(yx)
def run_problema_4(N):
    a, b = 1.0, 2.0
    c, d = 1.0, 2.0
    m, n = N, N

    def V_ana(x, y):
        return x * y * np.log(x * y)

    # Lado Derecho f(x, y) = x/y + y/x
    def f(x, y):
        return x / y + y / x

    # Condiciones de Frontera g(x, y)
    def g(x, y):
        if x == a: return 1 * y * np.log(1 * y) # V(1, y) = y*ln(y)
        if x == b: return 2 * y * np.log(2 * y) # V(2, y) = 2y*ln(2y)
        if y == c: return x * 1 * np.log(x * 1) # V(x, 1) = x*ln(x)
        if y == d: return x * 2 * np.log(x * 2) # V(x, 2) = 2x*ln(2x)
        return 0

    return solve_pde_fourier(m, n, a, b, c, d, f, g, V_ana)

# Diccionario de problemas para el constructor
PROBLEMS = {
    1: run_problema_1,
    2: run_problema_2,
    3: run_problema_3,
    4: run_problema_4
}

In [ ]:
# constructor.py
import os
import time
import pandas as pd
import numpy as np
from solver import PROBLEMS

# Definiciones
MALLAS = [5, 10, 20, 50, 100]
NUM_PROBLEMAS = 4
RUTA_BASE = "resultados"
TIEMPOS_FILE = os.path.join(RUTA_BASE, "tiempos_ejecucion.dat")

def setup_directorios():
    """Crea la estructura de carpetas de resultados."""
    if os.path.exists(RUTA_BASE):
        # Opcional: Limpiar o mover carpeta existente
        pass
    os.makedirs(RUTA_BASE, exist_ok=True)
    for p_id in range(1, NUM_PROBLEMAS + 1):
        for N in MALLAS:
            carpeta = os.path.join(RUTA_BASE, f"problema_{p_id}", f"{N}x{N}")
            os.makedirs(carpeta, exist_ok=True)

def guardar_resultados(df, p_id, N):
    """Guarda el DataFrame de resultados en el archivo .dat."""
    carpeta = os.path.join(RUTA_BASE, f"problema_{p_id}", f"{N}x{N}")
    filepath = os.path.join(carpeta, f"resultados_{N}x{N}.dat")

    # Formato de números cortos (cuatro cifras significativas)
    float_format = '%.4g'

    # Escribir el archivo
    with open(filepath, 'w') as f:
        # Escribir encabezados
        f.write(df.to_csv(index=False, sep='\t', header=True).split('\n')[0] + '\n')
        # Escribir datos formateados
        df.to_csv(f, index=False, sep='\t', header=False, float_format=float_format)

def ejecutar_simulacion():
    """Implementa el doble ciclo, mide el tiempo y guarda los datos."""
    print("Iniciando la simulación y generación de datos...")

    # Tabla para almacenar tiempos
    tiempos_data = {f"P{i}": [] for i in range(1, NUM_PROBLEMAS + 1)}
    tiempos_data["Malla"] = [f"{N}x{N}" for N in MALLAS]

    setup_directorios()

    for p_id in range(1, NUM_PROBLEMAS + 1):
        print(f"\n--- Procesando Problema {p_id} ---")
        run_func = PROBLEMS[p_id]

        for N in MALLAS:
            print(f"  Malla {N}x{N}...")

            # Medición de tiempo de ejecución
            start_time = time.perf_counter()
            resultados_df = run_func(N)
            end_time = time.perf_counter()

            tiempo_ms = (end_time - start_time) * 1000 # Tiempo en milisegundos
            tiempos_data[f"P{p_id}"].append(tiempo_ms)

            # Guardar resultados
            guardar_resultados(resultados_df, p_id, N)
            print(f"    Tiempo: {tiempo_ms:.2f} ms")


    # Guardar tabla de tiempos
    tiempos_df = pd.DataFrame(tiempos_data)
    tiempos_df = tiempos_df.set_index("Malla")

    # Formato de números cortos
    float_format = '%.4g'

    with open(TIEMPOS_FILE, 'w') as f:
        # Escribir encabezados
        f.write(tiempos_df.to_csv(sep='\t', header=True).split('\n')[0] + '\n')
        # Escribir datos formateados
        tiempos_df.to_csv(f, sep='\t', header=False, float_format=float_format)

    print(f"\nSimulación completa. Tiempos guardados en: {TIEMPOS_FILE}")
    print("Datos de resultados guardados en la carpeta 'resultados'.")

if __name__ == "__main__":
    ejecutar_simulacion()

Iniciando la simulación y generación de datos...

--- Procesando Problema 1 ---
  Malla 5x5...
    Tiempo: 15.68 ms
  Malla 10x10...
    Tiempo: 1.54 ms
  Malla 20x20...
    Tiempo: 4.77 ms
  Malla 50x50...
    Tiempo: 93.12 ms
  Malla 100x100...
    Tiempo: 136.11 ms

--- Procesando Problema 2 ---
  Malla 5x5...
    Tiempo: 1.12 ms
  Malla 10x10...
    Tiempo: 1.18 ms
  Malla 20x20...
    Tiempo: 4.10 ms
  Malla 50x50...
    Tiempo: 27.08 ms
  Malla 100x100...
    Tiempo: 111.07 ms

--- Procesando Problema 3 ---
  Malla 5x5...
    Tiempo: 1.34 ms
  Malla 10x10...
    Tiempo: 1.12 ms
  Malla 20x20...
    Tiempo: 4.46 ms
  Malla 50x50...
    Tiempo: 22.34 ms
  Malla 100x100...


/content/solver.py:99: RuntimeWarning: divide by zero encountered in divide
  error_rel = error_abs / np.abs(V_ana)


    Tiempo: 97.52 ms

--- Procesando Problema 4 ---
  Malla 5x5...
    Tiempo: 1.25 ms
  Malla 10x10...
    Tiempo: 1.33 ms
  Malla 20x20...
    Tiempo: 4.16 ms
  Malla 50x50...
    Tiempo: 26.65 ms
  Malla 100x100...
    Tiempo: 108.94 ms

Simulación completa. Tiempos guardados en: resultados/tiempos_ejecucion.dat
Datos de resultados guardados en la carpeta 'resultados'.


In [ ]:
# graficador.py
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Definiciones
MALLAS = [5, 10, 20, 50, 100]
MALLA_MAXIMA = MALLAS[-1] # 100
NUM_PROBLEMAS = 4
RUTA_BASE = "resultados"
RUTA_GRAFICOS = "graficos"
TIEMPOS_FILE = os.path.join(RUTA_BASE, "tiempos_ejecucion.dat")

def setup_directorios_graficos():
    """Crea la estructura de carpetas de gráficos."""
    os.makedirs(RUTA_GRAFICOS, exist_ok=True)
    for p_id in range(1, NUM_PROBLEMAS + 1):
        os.makedirs(os.path.join(RUTA_GRAFICOS, f"problema_{p_id}"), exist_ok=True)

def graficar_tiempos():
    """Genera la gráfica de comparación 2D de Tiempos de Ejecución vs. Tamaño de Malla."""
    print("Generando gráfica de tiempos de ejecución...")

    try:
        # La primera columna es el índice (Malla)
        tiempos_df = pd.read_csv(TIEMPOS_FILE, sep='\t', index_col=0)
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo de tiempos en {TIEMPOS_FILE}. Ejecute constructor.py primero.")
        return

    malla_labels = [f"{N}x{N}" for N in MALLAS]
    malla_size = np.array(MALLAS)

    plt.figure(figsize=(10, 6))

    for p_id in range(1, NUM_PROBLEMAS + 1):
        col_name = f"P{p_id}"
        plt.plot(malla_size, tiempos_df[col_name].values, marker='o', label=f'Problema {p_id}')

    plt.title('Tiempos de Ejecución vs. Tamaño de Malla (Método de Diferencias Finitas)')
    plt.xlabel('Tamaño de Malla N (NxN)')
    plt.ylabel('Tiempo de Ejecución (ms)')
    plt.xticks(malla_size, malla_labels)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.tight_layout()

    filepath = os.path.join(RUTA_GRAFICOS, "tiempos_ejecucion_comparacion.png")
    plt.savefig(filepath)
    plt.close()
    print(f"Gráfica de tiempos guardada en: {filepath}")

def graficar_resultados_3d():
    """Genera las gráficas 3D de Solución Numérica y Analítica para la malla máxima."""
    print(f"\nGenerando gráficas 3D para la malla máxima ({MALLA_MAXIMA}x{MALLA_MAXIMA})...")

    for p_id in range(1, NUM_PROBLEMAS + 1):
        print(f"  Procesando Problema {p_id}...")

        # Ruta del archivo de resultados de la malla máxima
        carpeta = os.path.join(RUTA_BASE, f"problema_{p_id}", f"{MALLA_MAXIMA}x{MALLA_MAXIMA}")
        filepath = os.path.join(carpeta, f"resultados_{MALLA_MAXIMA}x{MALLA_MAXIMA}.dat")

        try:
            # Los archivos .dat usan tabulador como separador
            df = pd.read_csv(filepath, sep='\t')
        except FileNotFoundError:
            print(f"    Error: Archivo de datos no encontrado para el Problema {p_id}. Saltando.")
            continue

        # Preparar los datos para la gráfica 3D (grilla)
        # N es el número total de intervalos, N-1 es el número de puntos interiores en una dirección
        N_pts = MALLA_MAXIMA - 1
        X = df['x'].values.reshape(N_pts, N_pts)
        Y = df['y'].values.reshape(N_pts, N_pts)

        V_num = df['V_num'].values.reshape(N_pts, N_pts)
        V_ana = df['V_ana'].values.reshape(N_pts, N_pts)

        # Generar Gráfica 3D 1: Solución Numérica
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        ax.plot_surface(X, Y, V_num, cmap='viridis', edgecolor='none')
        ax.set_title(f'P{p_id}: Solución Numérica (MDF, {MALLA_MAXIMA}x{MALLA_MAXIMA})')
        ax.set_xlabel('Coordenada X')
        ax.set_ylabel('Coordenada Y')
        ax.set_zlabel('V Numérica')

        grafico_path = os.path.join(RUTA_GRAFICOS, f"problema_{p_id}", f"P{p_id}_V_Num_{MALLA_MAXIMA}x{MALLA_MAXIMA}.png")
        plt.savefig(grafico_path)
        plt.close(fig)
        print(f"    Guardado V Numérica en: {grafico_path}")

        # Generar Gráfica 3D 2: Solución Analítica
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        ax.plot_surface(X, Y, V_ana, cmap='plasma', edgecolor='none')
        ax.set_title(f'P{p_id}: Solución Analítica ({MALLA_MAXIMA}x{MALLA_MAXIMA})')
        ax.set_xlabel('Coordenada X')
        ax.set_ylabel('Coordenada Y')
        ax.set_zlabel('V Analítica')

        grafico_path = os.path.join(RUTA_GRAFICOS, f"problema_{p_id}", f"P{p_id}_V_Ana_{MALLA_MAXIMA}x{MALLA_MAXIMA}.png")
        plt.savefig(grafico_path)
        plt.close(fig)
        print(f"    Guardado V Analítica en: {grafico_path}")


def main():
    setup_directorios_graficos()
    graficar_tiempos()
    graficar_resultados_3d()
    print("\nVisualización completa. Gráficas guardadas en la carpeta 'graficos'.")

if __name__ == "__main__":
    main()

Generando gráfica de tiempos de ejecución...
Gráfica de tiempos guardada en: graficos/tiempos_ejecucion_comparacion.png

Generando gráficas 3D para la malla máxima (100x100)...
  Procesando Problema 1...
    Guardado V Numérica en: graficos/problema_1/P1_V_Num_100x100.png
    Guardado V Analítica en: graficos/problema_1/P1_V_Ana_100x100.png
  Procesando Problema 2...
    Guardado V Numérica en: graficos/problema_2/P2_V_Num_100x100.png
    Guardado V Analítica en: graficos/problema_2/P2_V_Ana_100x100.png
  Procesando Problema 3...
    Guardado V Numérica en: graficos/problema_3/P3_V_Num_100x100.png
    Guardado V Analítica en: graficos/problema_3/P3_V_Ana_100x100.png
  Procesando Problema 4...
    Guardado V Numérica en: graficos/problema_4/P4_V_Num_100x100.png
    Guardado V Analítica en: graficos/problema_4/P4_V_Ana_100x100.png

Visualización completa. Gráficas guardadas en la carpeta 'graficos'.


In [ ]:
# graficador_mallas.py
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --- Configuraciones ---
MALLAS = [5, 10, 20, 50, 100]
NUM_PROBLEMAS = 4
RUTA_BASE = "resultados"
RUTA_GRAFICOS = "graficos"
COMPARACION_MALLAS_DIR = os.path.join(RUTA_GRAFICOS, "comparacion_mallas")

def setup_directorios_comparacion():
    """Crea la carpeta para las gráficas de comparación de mallas."""
    os.makedirs(COMPARACION_MALLAS_DIR, exist_ok=True)

def graficar_mallas_comparacion():
    """
    Genera un único plot con subgráficas 3D para comparar la solución numérica
    en las diferentes mallas para cada problema.
    """
    print("\nGenerando gráficas de comparación de mallas (3D)...")

    setup_directorios_comparacion()

    # Iterar sobre cada problema
    for p_id in range(1, NUM_PROBLEMAS + 1):
        print(f"  Procesando Problema {p_id}...")

        # 5 mallas -> 5 subgráficas. Usamos 1 fila y 5 columnas para que sea legible.
        fig = plt.figure(figsize=(25, 6))
        fig.suptitle(f'Problema {p_id}: Convergencia de la Solución Numérica (MDF)', fontsize=16)

        datos_cargados = True

        # Iterar sobre cada tamaño de malla
        for idx, N in enumerate(MALLAS):
            N_label = f"{N}x{N}"

            # Construir la ruta al archivo
            carpeta = os.path.join(RUTA_BASE, f"problema_{p_id}", N_label)
            filepath = os.path.join(carpeta, f"resultados_{N}x{N}.dat")

            try:
                # Cargar datos
                df = pd.read_csv(filepath, sep='\t')
            except FileNotFoundError:
                print(f"    Advertencia: Archivo de datos no encontrado para P{p_id}, Malla {N_label}. Saltando.")
                datos_cargados = False
                continue

            # Preparar datos 3D
            N_pts = N - 1 # Número de puntos interiores

            # Asegurarse de que el número de puntos sea consistente
            if len(df) != N_pts * N_pts:
                print(f"    Error de datos: El número de filas en {N_label} no coincide con el esperado.")
                datos_cargados = False
                continue

            # Remodelar las coordenadas y la solución
            X = df['x'].values.reshape(N_pts, N_pts)
            Y = df['y'].values.reshape(N_pts, N_pts)
            V_num = df['V_num'].values.reshape(N_pts, N_pts)

            # Crear el subplot 3D
            ax = fig.add_subplot(1, 5, idx + 1, projection='3d')

            # Graficar la superficie
            surf = ax.plot_surface(X, Y, V_num, cmap='viridis', edgecolor='none')

            # Configuración de ejes y título
            ax.set_title(f'{N_label}', fontsize=12)
            ax.set_xlabel('X')
            ax.set_ylabel('Y')

            # Para evitar que el eje Z cambie drásticamente entre plots, se puede fijar el rango
            # Si no se fija, matplotlib ajustará el rango a los datos de cada subplot

        if datos_cargados:
            # Ajustar layout y guardar la figura
            plt.tight_layout(rect=[0, 0, 1, 0.95]) # Deja espacio para el suptitle

            filepath_guardar = os.path.join(COMPARACION_MALLAS_DIR, f"P{p_id}_Mallas_Comparacion.png")
            plt.savefig(filepath_guardar)
            plt.close(fig)
            print(f"  Gráfica de comparación guardada para P{p_id} en: {filepath_guardar}")
        else:
            plt.close(fig)

    print("\nGeneración de gráficas de mallas completa.")


if __name__ == "__main__":
    graficar_mallas_comparacion()